# Stage 1 — Finance Non-Instruction Fine-Tuning
This standalone Colab notebook adapts a Qwen2.5 0.5B model to finance and banking language using raw domain text. It includes preprocessing, QLoRA training, evaluation, adapter saving, and a Finance-only ipywidgets demo.

In [1]:
# Install pinned, Colab-compatible packages
!pip install -q "unsloth[colab-new]" "trl>=0.18,<0.26" "transformers>=4.51,<5" datasets accelerate bitsandbytes ipywidgets pandas matplotlib


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 125.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 122.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from pathlib import Path
import json, warnings, torch
from datasets import Dataset
warnings.filterwarnings("ignore")

DOMAIN = "Finance FAQ Assistant"
DOMAIN_OPTIONS = [DOMAIN]
BASE_MODEL = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 1024
DTYPE = None
LOAD_IN_4BIT = True
SEED = 42

print("Domain:", DOMAIN)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")


Domain: Finance FAQ Assistant
GPU: Tesla T4


In [5]:
# Locate extracted data directory (no ZIP upload required).
candidates = [
    Path("../data"),                # local run from notebooks/
    Path("./data"),                 # local run from project root
    Path("/content/data"),          # Colab with uploaded/extracted data folder
    Path("/data"),                  # absolute data mount
    Path("/content/finance_stage1_data"),  # backward compatibility
    Path("/content/finance_stage2_data"),
    Path("/content/finance_stage3_data"),
]

DATA_DIR = next((p for p in candidates if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find extracted data folder. Expected one of: " + ", ".join(str(p) for p in candidates)
    )

DATA_DIR = DATA_DIR.resolve()
print("Using DATA_DIR:", DATA_DIR)
print("Available files:", sorted(p.name for p in DATA_DIR.glob("*") if p.is_file()))


Using DATA_DIR: /content/data
Available files: ['non_instruction_data.txt']


## Load and chunk raw finance text

In [6]:
raw_text=(DATA_DIR/"non_instruction_data.txt").read_text(encoding="utf-8")
paragraphs=[p.strip() for p in raw_text.split("\n\n") if p.strip()]
chunks=[]
buffer=""
for p in paragraphs:
    if len(buffer)+len(p)<1800: buffer=(buffer+"\n\n"+p).strip()
    else: chunks.append({"text":buffer}); buffer=p
if buffer: chunks.append({"text":buffer})
train_dataset=Dataset.from_list(chunks)
print("Paragraphs:",len(paragraphs),"Chunks:",len(train_dataset))
print(train_dataset[0]["text"][:600])

Paragraphs: 56 Chunks: 15
A savings account is one of the most fundamental banking products offered by financial institutions. It allows individuals to deposit money and earn interest over time. Savings accounts are designed for individuals who want to keep their money safe while earning a modest return. The interest rate on savings accounts typically ranges from 3% to 7% per annum, depending on the bank and the account balance maintained.

A current account, also known as a checking account, is primarily designed for businesses and individuals who perform high-frequency transactions. Unlike savings accounts, current a


In [7]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
print("Model and QLoRA adapter initialized.")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2026.7.2 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


Model and QLoRA adapter initialized.


## Train Stage 1

In [8]:
from trl import SFTTrainer, SFTConfig
OUTPUT_DIR=Path("/content/finance_adapters/stage1_non_instruction")
args=SFTConfig(output_dir=str(OUTPUT_DIR),dataset_text_field="text",max_length=MAX_SEQ_LENGTH,per_device_train_batch_size=1,gradient_accumulation_steps=4,num_train_epochs=1,learning_rate=2e-4,logging_steps=1,save_strategy="no",report_to="none",optim="adamw_8bit",seed=SEED)
trainer=SFTTrainer(model=model,tokenizer=tokenizer,train_dataset=train_dataset,args=args)
result=trainer.train()
model.save_pretrained(str(OUTPUT_DIR)); tokenizer.save_pretrained(str(OUTPUT_DIR))
print("Saved:",OUTPUT_DIR)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/15 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 15 | Num Epochs = 1 | Total steps = 4
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)


Step,Training Loss
1,1.969800
2,2.107100
3,2.058800
4,1.833500


Saved: /content/finance_adapters/stage1_non_instruction


## Step 5 - Base model evaluation (before instruction fine-tuning)

In [9]:
import re
from pathlib import Path
import pandas as pd
from IPython.display import display

# Step 5 uses these 10 questions for the base-model table.
EVAL_QUESTIONS = [
    "What is a savings account?",
    "What is the difference between savings and current account?",
    "What documents are required for KYC?",
    "How do I transfer money using NEFT?",
    "What is RTGS and when should I use it?",
    "What is a credit score and why does it matter?",
    "How can I apply for a personal loan?",
    "What should I do if I receive a suspicious OTP request?",
    "My EMI payment failed. What should I do?",
    "How can I improve my credit score?",
]

def ask_model(question, max_new_tokens=220, temperature=0.2):
    prompt = f"Finance domain context question: {question}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=max(temperature, 0.01),
        top_p=0.9,
        repetition_penalty=1.08,
        pad_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return text.strip()

def diagnose_problem(question, answer):
    q = question.lower()
    a = answer.lower().strip()
    words = re.findall(r"[a-zA-Z]+", a)

    if len(words) < 22:
        return "Answer is too brief and misses important finance-specific detail."

    vague_markers = ["contact your bank", "check with your bank", "it depends", "usually", "generally"]
    if any(marker in a for marker in vague_markers):
        return "Answer is generic and not specific enough to finance workflows or policy context."

    if "otp" in q and not any(k in a for k in ["never share", "fraud", "report", "block", "cybercrime"]):
        return "Safety guidance is incomplete for fraud/OTP risk handling."

    if any(k in q for k in ["how do i", "how can i", "what should i do"]) and not any(
        k in a for k in ["step", "log in", "portal", "app", "documents", "submit", "contact support", "immediately"]
    ):
        return "Procedural guidance is weak; answer does not provide actionable steps."

    domain_terms = [
        "kyc", "neft", "rtgs", "imps", "upi", "emi", "cibil", "interest", "loan", "deposit", "bank", "account"
    ]
    if not any(term in a for term in domain_terms):
        return "Answer lacks domain terminology and appears too general."

    return "Answer is partially correct but can be more specific, complete, and policy-grounded."

def resolve_reports_dir():
    candidates = []
    if "DATA_DIR" in globals():
        candidates.append(Path(DATA_DIR).resolve().parent / "reports")
    candidates.extend([Path.cwd() / "reports", Path("/content/reports"), Path("./reports"), Path("../reports")])

    for p in candidates:
        try:
            p.mkdir(parents=True, exist_ok=True)
            probe = p / ".write_probe"
            probe.write_text("ok", encoding="utf-8")
            probe.unlink()
            return p.resolve()
        except Exception:
            continue
    raise RuntimeError("Could not create a writable reports directory.")

FastLanguageModel.for_inference(model)
base_answers = {q: ask_model(q) for q in EVAL_QUESTIONS}
problems = {q: diagnose_problem(q, base_answers[q]) for q in EVAL_QUESTIONS}

df_base = pd.DataFrame([
    {"Question": q, "Base Model Answer": base_answers[q], "Problem": problems[q]}
    for q in EVAL_QUESTIONS
])
display(df_base)

# Export Step 5 table to reports/base_model_evaluation.md
reports_dir = resolve_reports_dir()
out_path = reports_dir / "base_model_evaluation.md"

lines = [
    "# Base Model Evaluation (Finance FAQ Assistant)",
    "",
    "This report captures baseline behavior of the original base model before instruction fine-tuning.",
    "",
    "## Evaluation Setup",
    "",
    "- Domain: Finance FAQ Assistant",
    "- Dataset reference: `data/instruction_dataset.jsonl`",
    "- Number of evaluation questions: 10",
    "- Goal: Identify generic, incomplete, or non-domain-specific behavior in base model outputs.",
    "",
    "## Baseline Results",
    "",
    "| Question | Base Model Answer | Problem |",
    "| --- | --- | --- |",
]
for q in EVAL_QUESTIONS:
    ans = " ".join(base_answers[q].split()).replace("|", "\\|")
    prob = " ".join(problems[q].split()).replace("|", "\\|")
    lines.append(f"| {q} | {ans} | {prob} |")

lines += [
    "",
    "## Summary",
    "",
    "- Typical baseline gaps observed: Generic language, limited procedural detail, and missing safety specifics in some responses.",
    "- Most common failure type (generic, inaccurate, unsafe, incomplete): Generic and incomplete responses.",
    "- Key areas to improve through SFT: Domain terminology precision, step-by-step instructions, and safer financial guidance.",
]

out_path.write_text("\n".join(lines), encoding="utf-8")
print("Step 5 report written:", out_path)

,Question,Base Model Answer,Problem
0,What is a savings account?,Savings accounts are deposit accounts that off...,Answer is generic and not specific enough to f...
1,What is the difference between savings and cur...,Savings are deposits that can be withdrawn at ...,Answer is generic and not specific enough to f...
2,What documents are required for KYC?,KYC is a process of identifying the beneficial...,Answer is partially correct but can be more sp...
3,How do I transfer money using NEFT?,Transfer of funds from a bank account to anoth...,Answer is partially correct but can be more sp...
4,What is RTGS and when should I use it?,Real-time transfer of funds (RTF) refers to th...,Answer is partially correct but can be more sp...
5,What is a credit score and why does it matter?,A credit score is a numerical representation o...,Answer is partially correct but can be more sp...
6,How can I apply for a personal loan?,"To apply for a personal loan, you will need to...",Answer is partially correct but can be more sp...
7,What should I do if I receive a suspicious OTP...,"If you receive an OTP request, it is important...",Answer is partially correct but can be more sp...
8,My EMI payment failed. What should I do?,"If your EMIs have been cancelled, you can cont...",Procedural guidance is weak; answer does not p...
9,How can I improve my credit score?,Improving your credit score is a process that ...,Answer is partially correct but can be more sp...


Step 5 report written: /content/reports/base_model_evaluation.md


## Interactive demo

In [10]:
# Finance-only ipywidgets demo
import html, ipywidgets as widgets
from IPython.display import display, HTML

FastLanguageModel.for_inference(model)

domain_dd = widgets.Dropdown(options=DOMAIN_OPTIONS, value=DOMAIN, description="Domain", style={"description_width":"100px"}, layout=widgets.Layout(width="580px"))
question_box = widgets.Textarea(value="What is the difference between NEFT and RTGS?", description="Question", style={"description_width":"100px"}, layout=widgets.Layout(width="900px", height="110px"))
max_tokens = widgets.IntSlider(value=220, min=80, max=400, step=20, description="Max tokens", style={"description_width":"100px"}, layout=widgets.Layout(width="580px"))
temperature = widgets.FloatSlider(value=0.2, min=0.1, max=0.8, step=0.05, description="Temperature", style={"description_width":"100px"}, layout=widgets.Layout(width="580px"))
button = widgets.Button(description="Generate Finance Answer", button_style="primary", layout=widgets.Layout(width="260px", height="44px"))
status = widgets.HTML()
answer = widgets.HTML(value="<div style='padding:18px;border:1px solid #cbd5e1;border-radius:14px;background:white'>Answer will appear here.</div>")

def render(text):
    return f"<div style='padding:22px;border:1px solid #bfdbfe;border-radius:16px;background:linear-gradient(180deg,#fff,#eff6ff);line-height:1.7'><b style='color:#1e3a8a'>Finance Assistant Answer</b><hr>{html.escape(text).replace(chr(10),'<br>')}</div>"

def on_click(_):
    q=question_box.value.strip()
    if not q:
        answer.value=render("Please enter a finance-related question.")
        return
    button.disabled=True; status.value="<b style='color:#2563eb'>Generating...</b>"
    try:
        torch.cuda.empty_cache()
        answer.value=render(ask_model(q, max_new_tokens=max_tokens.value, temperature=temperature.value))
        status.value="<b style='color:#15803d'>Completed</b>"
    except Exception as exc:
        answer.value=render(f"{type(exc).__name__}: {exc}")
        status.value="<b style='color:#b91c1c'>Failed</b>"
    finally:
        button.disabled=False
button.on_click(on_click)

display(HTML("""<div style='background:linear-gradient(135deg,#0f172a,#1d4ed8);color:white;padding:26px;border-radius:18px;text-align:center;margin-bottom:18px'><h1>Enterprise Finance FAQ Assistant</h1><p>Standalone fine-tuning stage demonstration</p></div>"""))
display(domain_dd, max_tokens, temperature, question_box, button, status, answer)


Dropdown(description='Domain', layout=Layout(width='580px'), options=('Finance FAQ Assistant',), style=Descrip…

IntSlider(value=220, description='Max tokens', layout=Layout(width='580px'), max=400, min=80, step=20, style=S…

FloatSlider(value=0.2, description='Temperature', layout=Layout(width='580px'), max=0.8, min=0.1, step=0.05, s…

Textarea(value='What is the difference between NEFT and RTGS?', description='Question', layout=Layout(height='…

Button(button_style='primary', description='Generate Finance Answer', layout=Layout(height='44px', width='260p…

HTML(value='')

HTML(value="<div style='padding:18px;border:1px solid #cbd5e1;border-radius:14px;background:white'>Answer will…